# UrbanPulse - Bengaluru Crime & Safety EDA
Smart City Transparency Platform Crime Analytics Module

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import xml.etree.ElementTree as ET
import folium
import json

## 1. Police Station KML Locations Extraction (150 Locations)

In [ ]:
kml_files = glob.glob("../../data/kml/*.kml")
rows = []
for kml_path in kml_files:
    tree = ET.parse(kml_path)
    root = tree.getroot()
    names = [elem.text.strip() for elem in root.iter() if elem.tag.split("}")[-1] == "name" and elem.text]
    layer_name = names[0] if names else "Police Station"
    for elem in root.iter():
        if elem.tag.split("}")[-1] == "coordinates":
            text = (elem.text or "").strip()
            for item in text.replace("\n", " ").split():
                parts = item.split(",")
                if len(parts) >= 2:
                    try:
                        rows.append({"Police Station": layer_name, "Latitude": float(parts[1]), "Longitude": float(parts[0])})
                    except ValueError:
                        pass
police_df = pd.DataFrame(rows).drop_duplicates(subset=["Latitude", "Longitude"]).reset_index(drop=True)
print(f"Total unique mapped locations: {len(police_df)}")
police_df.head()

## 2. City Crime Risk Score Formula (UrbanPulse Algorithm)
- 60% Crime Volume Score + 40% Severity Score
- High Risk (>= 70), Medium Risk (40-69), Low Risk (< 40)

In [ ]:
historical_stats = {
    "Total Recorded Crimes": 35971,
    "Fatal Crimes": 5437,
    "Non-Fatal Crimes": 30534,
    "Overall Fatal Percentage": 15.11,
    "Highest Risk Year": 2016,
    "Highest Risk Score": 71.70,
    "Risk Level": "High Risk"
}
print(historical_stats)